# Dirac domain-wall spectra on the sphere: reproducible computational study

This notebook reruns the pole-regular Jacobi–Galerkin simulations and regenerates the CSV tables and figures. In Google Colab, the setup cell clones the public GitHub repository and installs its listed dependencies.

**Local use:** run this notebook from the repository directory. If the files are not present, the setup cell clones the repository into the current environment.


In [ ]:
from pathlib import Path
import os, sys, subprocess

ROOT = Path.cwd()
if not (ROOT / "run_study.py").is_file():
    ROOT = Path("/content/dirac-domain-wall-sphere") if "google.colab" in sys.modules else Path.cwd() / "dirac-domain-wall-sphere"
    if not (ROOT / "run_study.py").is_file():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ozalloum/dirac-domain-wall-sphere.git", str(ROOT)], check=True)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Repository directory:", ROOT)


In [ ]:
%pip install -q -r requirements.txt


## Run the full simulation suite

The full run may take a few minutes in a standard Colab session. It writes fresh results under `data/` and `figures/` in the extracted package directory.

In [ ]:
from run_study import run

summary = run(ROOT, nmain=112, nq=1000, quick=False)
print("Run summary")
for key, value in summary.items():
    print(f"- {key}: {value}")


## Inspect the raw CSV results

In [ ]:
import csv

def preview_csv(filename, n=8):
    with (ROOT / "data" / filename).open(newline="") as stream:
        rows = list(csv.DictReader(stream))
    print(filename, "—", len(rows), "rows")
    for row in rows[:n]:
        print(row)

preview_csv("exact_spectrum_validation.csv", 6)
preview_csv("single_wall_chiral_branch.csv")
preview_csv("symmetric_two_wall_gap.csv", 6)
preview_csv("excited_pair_separation.csv")
preview_csv("independent_shooting_benchmark.csv")
preview_csv("solver_cost_benchmark.csv")


## Independent method check and runtime measurements

The shooting CSV compares the Jacobi–Galerkin branch with an independent two-sided ODE integration. The cost CSV reports host-specific median timings and dense-matrix storage; rerunning on Colab may produce different runtime values.

In [ ]:
preview_csv("independent_shooting_benchmark.csv")
preview_csv("solver_cost_benchmark.csv")


## Display the generated vector figures


In [ ]:
from IPython.display import SVG, display

for name in [
    "single_wall_chiral_branch.svg",
    "symmetric_two_wall_gap.svg",
    "excited_double_wall_pair.svg",
    "excited_pair_separation.svg",
    "localized_eigenfunction_density.svg",
    "basis_convergence.svg",
]:
    print(name)
    display(SVG(filename=str(ROOT / "figures" / name)))


## Try another smooth axisymmetric mass profile

The function below demonstrates the reusable solver on a single sector. The default package tables and figures use only the two cosine profiles described in the paper. This extra calculation is illustrative; it is not part of the reported results.

In [ ]:
import numpy as np
from sphere_dirac_galerkin import PoleRegularBasis

basis = PoleRegularBasis.build(nu=0.5, size=64, quadrature_order=600)
epsilon = 0.03
mass_profile = lambda theta: np.cos(theta) + 0.15 * np.sin(theta) ** 2
spectrum = basis.solve(epsilon, mass_profile, radius=1.0)
print("Eight eigenvalues nearest zero:", spectrum[np.argsort(np.abs(spectrum))[:8]])
